# July NDVI for the San Carlos Apache Reservation

Compare July observations in 2001–2003 and 2020–2022 using NASA AppEEARS. These are example comparison periods, not a test of the Gila River water-rights event.


In [1]:
# This portfolio example loads a new boundary, so no stored variables are needed.


You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries

### file structure
import os
import pathlib

### plotting
import hvplot.pandas
import hvplot.xarray

### dealing with spatial data
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
import pandas as pd

### earthpy
import earthpy
import earthpy.api.appeears as eaapp

## Exploring the AppEEARS API for NASA Earthdata access

Before you get started with the data download, you will need a free [NASA Earthdata account](https://urs.earthdata.nasa.gov/home) if you don’t have one already.

You will download MODIS NDVI data for the study period. MODIS is a multispectral instrument that measures red and NIR data (and so can be used for NDVI). There are two MODIS sensors on two different platforms: satellites Terra and Aqua.

You can learn more about MODIS datasets and the science they support [here](https://modis.gsfc.nasa.gov/).

Since we’re asking for a special download that only covers our study area, we can’t just find a link to the data. Instead, we have to negotiate with the data server. We’re doing this using the
[APPEEARS](https://appeears.earthdatacloud.nasa.gov/api/) API (Application Programming Interface). The API makes it possible for you to request data using code. You can use code from the `earthpy` library to handle the API request.

Often when we want to do something more complex in coding, we find an example and modify it. This download code is already almost a working example. Your task will be to:   

- Replace the start and end dates in the task parameters. Download data from July, when greenery is at its peak in the Northern Hemisphere.  
- Replace the year range. You should get two 3 year periods so you can see the change.  
- Replace `gdf` with the name of your site geodataframe.  
- Enter your NASA Earthdata username and password when prompted at the top of your screen.


**Note:** It can take some time for Appeears to process your request - anything from a few minutes to a few hours depending on how busy they are. You can check your progress by:

1. Going to the [Appeears webpage](https://appeears.earthdatacloud.nasa.gov/)  
2. Clicking the `Explore` tab  
3. Logging in with your Earthdata account

The portfolio example uses the San Carlos Reservation boundary from the Census TIGERweb service (GEOID 3355R). July is the requested comparison month, not necessarily the peak-greenness month at every site. AppEEARS requests a bounding rectangle around the supplied boundary; clip the downloaded rasters during analysis. NASA credentials are entered only through the library prompt, never in this notebook.


In [3]:
### get the San Carlos Reservation boundary from the U.S. Census Bureau
import requests

boundary_url = (
    'https://tigerweb.geo.census.gov/arcgis/rest/services/'
    'TIGERweb/AIANNHA/MapServer/2/query'
)
response = requests.get(
    boundary_url,
    params={
        'where': "NAME = 'San Carlos Reservation'",
        'outFields': 'NAME,GEOID',
        'returnGeometry': 'true',
        'outSR': 4326,
        'f': 'geojson'
    },
    timeout=60
)
response.raise_for_status()
site_gdf = gpd.GeoDataFrame.from_features(response.json()['features'], crs='EPSG:4326')

### check that the intended boundary was returned
assert len(site_gdf) == 1
print(site_gdf[['NAME', 'GEOID']])

### make a separate home for the portfolio downloads
project = earthpy.Project(dirname='san-carlos-portfolio-ndvi')
print(project.project_dir)


                     NAME  GEOID
0  San Carlos Reservation  3355R
C:\Users\gekek\AppData\Local\earth-analytics\earth-analytics\san-carlos-portfolio-ndvi


In [ ]:
### choose two three-year periods and a descriptive download key
# Change these years to match the question for your portfolio location.
year_ranges = [[2001, 2003], [2020, 2022]]
download_key = 'san-carlos-july-ndvi'

# check downloader
ndvi_downloader.login(interactive=True, override=True)
print('Earthdata login successful.')

ndvi_downloader.download_files(cache=False)

### request each period separately
for year_range in year_ranges:
    period_key = f'{download_key}-{year_range[0]}-{year_range[1]}'

    ### initialize AppEEARS for July MODIS NDVI
    ndvi_downloader = eaapp.AppeearsDownloader(
        download_key=period_key,
        project=project,
        product='MOD13Q1.061',
        layer='_250m_16_days_NDVI',
        start_date='07-01',
        end_date='07-31',
        recurring=True,
        year_range=year_range,
        polygon=site_gdf
    )

    # This Earthpy version normally shares one task ID across all downloads.
    # Keep each period's task ID separate so it can resume the correct request.
    ndvi_downloader.task_id_path = project.project_dir / f'{period_key}-task-id.txt'
    if ndvi_downloader.task_id_path.exists():
        ndvi_downloader._task_id = ndvi_downloader.task_id_path.read_text().strip()
    else:
        ndvi_downloader._task_id = None

    print(f'Downloading July NDVI for {year_range[0]}-{year_range[1]}')
    # Do not treat an incomplete download folder as a finished download.
    # Rerunning this cell can request and download the files again.
    ndvi_downloader.download_files(cache=False)

### save the project path and boundary for the wrangling notebook
# This is a separate IPython store variable from the original Gila River project.
portfolio_project = project
portfolio_gdf = site_gdf
%store portfolio_project portfolio_gdf


No stored credentials found for urs.earthdata.nasa.gov. Please log in.


After the downloads finish, return to Step 3A in notebook 03.
For the portfolio workflow, restore the new project and boundary first:

```python
%store -r portfolio_project portfolio_gdf
project = portfolio_project
gila_river_gdf = portfolio_gdf
```

The recursive NDVI filename pattern will find files from both period folders.
Use 2001–2003 and 2020–2022 in the plotting notebook for these example downloads, rather than the original longer periods.
For a different study area, replace `site_gdf` and update the project directory, download key, years, and plot labels.
